In [10]:
"""Expanded 12-model, 12-variant static DI ablation study.
Run in one notebook cell or as a .py file. Original outputs are never modified.
Classification: balanced accuracy; regression: thesis rAccuracy_tol.
"""
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier, GradientBoostingRegressor, RandomForestRegressor
from xgboost import XGBClassifier, XGBRegressor

ROOT = Path(r'C:\Users\suuser\Desktop\minlplib')
FINAL = ROOT / 'Areview' / 'combined_minlplib_qplib'
BB_RUN = FINAL / 'sklearn_all_runs_synthetic_prime_8016_20261021_014902_034048'
ROOT_RUN = FINAL / 'sklearn_root_synprime_8016_20261021_020512_941535'
SYN_DIR = ROOT / 'paperexperiments' / 'synthetic_prime_minimal_revision' / 'bb_separate_trees' / 'synthetic_prime_8016_fixed100_v1' / 'postrun_clean'
COMBINED_STATIC = FINAL / 'combined_minlplib_qplib_features_DI_64_clean.xlsx'
SYN_STATIC = SYN_DIR / 'synthetic_prime_DI_64_clean.xlsx'
COMBINED_BB_TARGET = FINAL / 'targets' / 'method2' / 'combined_minlplib_qplib' / 'targets_method2_relaxation_combined_minlplib_qplib_probe10.csv'
SYN_BB_TARGET = SYN_DIR / 'targets' / 'method2' / 'synthetic_prime' / 'targets_method2_relaxation_synthetic_prime_probe10.csv'
BB_SPLIT = BB_RUN / 'combined_train_test_split_manifest.csv'
ROOT_COMBINED_TARGET = ROOT_RUN / 'root_targets_combined.csv'
ROOT_SYN_TARGET = ROOT_RUN / 'root_targets_synthetic_prime.csv'
ROOT_COMBINED_SUMMARY = FINAL / 'summary_combined_minlplib_qplib_BB_clean.csv'
ROOT_SYN_SUMMARY = SYN_DIR / 'synthetic_prime_BB_summary.csv'
OUT = FINAL / ('ablation_12_models_extended_' + datetime.now().strftime('%Y%m%d_%H%M%S_%f'))
RANDOM_STATE = 42
EXCLUDE = {'QPLIB||QPLIB_2512','QPLIB||QPLIB_3402','QPLIB||QPLIB_3703'}
EXPECTED_COMBINED = 280
EXPECTED_SYN = 6069

# Four main comparisons plus two cross-model robustness checks.
# names intentionally match existing permutation-importance experiment labels.
PLAN = [
    ('BB_Combined', 'bb', 'combined', 'GBC', 'RFR', 'primary'),
    ('BB_Transfer', 'bb', 'transfer', 'XGB-C', 'RFR', 'primary'),
    ('Root_Combined', 'root', 'combined', 'XGB-C', 'XGB-R', 'primary'),
    ('Root_Transfer', 'root', 'transfer', 'GBC', 'GBR', 'primary'),
    ('Root_Combined_Robustness', 'root', 'combined', 'GBC', 'GBR', 'robustness'),
    ('Root_Transfer_Robustness', 'root', 'transfer', 'XGB-C', 'XGB-R', 'robustness'),
]

# These 64 names are copied from the user's actual all_feature_importance.csv.
OBJECTIVE_FEATURES = ['BoundsExist', 'PB_A0', 'PC_A0', 'PD_A0', 'PH_A0', 'PP_A0', 'PT_A0',
                      'eta_A0_over_n', 'lambda_max_A0_scaled', 'lambda_min_A0_scaled',
                      'rho_A0_over_n', 'theta_A0']
CONSTRAINT_FEATURES = ['h_CV_Lambda_max_scaled', 'h_CV_Lambda_min_scaled', 'h_CV_LogCond', 'h_CV_Rho', 'h_CV_Theta', 'h_CV_Xi',
'h_IQR_Lambda_max_scaled', 'h_IQR_Lambda_min_scaled', 'h_IQR_LogCond', 'h_IQR_Rho', 'h_IQR_Theta', 'h_IQR_Xi',
'h_Pout_Lambda_max_scaled', 'h_Pout_Lambda_min_scaled', 'h_Pout_Rho', 'h_Pout_Theta', 'h_Pout_Xi',
'h_avg_Lambda_max_scaled', 'h_avg_Lambda_min_scaled', 'h_avg_LogCond', 'h_avg_Rho', 'h_avg_Theta', 'h_avg_Xi',
'h_max_Lambda_max_scaled', 'h_max_Lambda_min_scaled', 'h_max_Rho', 'h_max_Theta', 'h_max_Xi',
'h_min_Lambda_max_scaled', 'h_min_Lambda_min_scaled', 'h_min_Rho', 'h_min_Theta', 'h_min_Xi',
'h_skewness_Lambda_max_scaled', 'h_skewness_Lambda_min_scaled', 'h_skewness_LogCond', 'h_skewness_Rho', 'h_skewness_Theta', 'h_skewness_Xi']
STRUCTURAL_FEATURES = ['comm_max', 'comm_mean', 'g0_Theta', 'g1_Theta', 'gB_PhiB', 'gC_PhiC', 'gD_PhiD',
                       'gH_PhiH', 'gP_PhiP', 'gT_PhiT', 'g_indef_Theta', 'g_linear_mass', 'g_overlap_Theta']
# BoundsExist is examined separately: objective-spectrum group does NOT remove it.
OBJECTIVE_SPECTRAL = [x for x in OBJECTIVE_FEATURES if x != 'BoundsExist']
# Pre-specified groups from real 64-feature schema (no test-score-based selection).
OBJECTIVE_EXTREME = ['lambda_max_A0_scaled', 'lambda_min_A0_scaled', 'rho_A0_over_n']
OBJECTIVE_SHAPE = [x for x in OBJECTIVE_SPECTRAL if x not in OBJECTIVE_EXTREME]
H_CENTRAL = [x for x in CONSTRAINT_FEATURES if x.startswith(('h_avg_', 'h_min_', 'h_max_'))]
H_DISPERSION = [x for x in CONSTRAINT_FEATURES if x not in H_CENTRAL]
ABLATIONS = {
    'Full_64': [],
    'No_objective_A0': OBJECTIVE_SPECTRAL,
    'No_constraint_h': CONSTRAINT_FEATURES,
    'No_structure_g_comm': STRUCTURAL_FEATURES,
    'No_BoundsExist': ['BoundsExist'],
    'No_objective_extremes': OBJECTIVE_EXTREME,
    'No_objective_shape': OBJECTIVE_SHAPE,
    'No_h_central': H_CENTRAL,
    'No_h_dispersion': H_DISPERSION,
    'No_objective_or_h': OBJECTIVE_SPECTRAL + CONSTRAINT_FEATURES,
    'Only_objective_and_bounds': [x for x in CONSTRAINT_FEATURES + STRUCTURAL_FEATURES],
    'Only_constraint_and_bounds': OBJECTIVE_SPECTRAL + STRUCTURAL_FEATURES,
}



def canon_combined(df):
    """Match the original B&B/root Combined key rules, including global id fallback."""
    df = df.copy()
    if 'instance_key' in df.columns:
        key = df['instance_key'].astype('string').str.strip()
    else:
        if 'instance_name' not in df.columns:
            raise ValueError('Combined table must contain instance_key or instance_name')
        names = (df['instance_name'].astype('string').str.strip()
                 .str.replace(r'\.pkl$', '', regex=True))
        if 'source' in df.columns:
            source = df['source'].astype('string').str.upper().str.strip()
        elif 'id' in df.columns:
            ids = df['id'].astype('string').str.strip()
            source = pd.Series(np.where(ids.str.startswith('MINLPLIB__', na=False), 'MINLPLIB',
                                np.where(ids.str.startswith('QPLIB__', na=False), 'QPLIB', '')),
                               index=df.index, dtype='string')
            if (source == '').any():
                examples = ids.loc[source == ''].head(10).tolist()
                raise ValueError(f'Could not infer Combined source from global id: {examples}')
        else:
            raise ValueError('Combined table requires instance_key, source, or global id')
        key = source + '||' + names
    if key.isna().any() or key.eq('').any():
        raise ValueError('Combined table contains blank instance identities')
    df['_key'] = key.astype(str).to_numpy()
    verify_unique(df, 'Combined')
    return df


def canon_synthetic(df, label, expected_keys=None):
    """Match stable original synthetic IDs exactly as in the root-only runner.

    The synthetic static Excel can contain a non-unique instance_index. Never use
    that as an identifier; choose an actual stable ID with overlap to target keys.
    """
    df = df.copy()
    candidates = [c for c in ('_key', 'instance_key', 'dataset_id', 'instance_id', 'id') if c in df.columns]
    if not candidates:
        raise ValueError(f'{label}: none of instance_key, dataset_id, instance_id, id found')
    best = None
    for c in candidates:
        raw = df[c].astype('string').str.strip().str.replace(r'^SYNTHETIC\|\|', '', regex=True)
        if raw.isna().any() or not raw.str.fullmatch(r'[0-9]+').all():
            continue
        keys = 'SYNTHETIC||' + raw.astype(str)
        if keys.duplicated().any():
            continue
        overlap = len(set(keys) & expected_keys) if expected_keys is not None else len(keys)
        if best is None or overlap > best[0]:
            best = (overlap, c, keys)
    if best is None or (expected_keys is not None and best[0] != len(expected_keys)):
        raise ValueError(f'{label}: cannot match all {len(expected_keys) if expected_keys is not None else "?"} expected synthetic keys; candidate overlap={None if best is None else best[:2]}')
    df['_key'] = best[2].to_numpy()
    verify_unique(df, label)
    print(f'{label}: matched via {best[1]}, overlap={best[0]}')
    return df


def verify_unique(df, label):
    if df['_key'].isna().any() or df['_key'].duplicated().any():
        bad = df.loc[df['_key'].duplicated(keep=False), '_key'].head(12).tolist()
        raise ValueError(label + ' has missing/duplicate keys: ' + str(bad))


def load_features(path, kind, expected_keys=None):
    raw = pd.read_excel(path)
    return canon_combined(raw) if kind=='combined' else canon_synthetic(raw, 'Synthetic static', expected_keys)


def load_targets(path, kind, ycol):
    raw = pd.read_csv(path)
    raw = canon_combined(raw) if kind=='combined' else canon_synthetic(raw, 'Synthetic targets')
    if kind == 'combined':
        raw = raw.loc[~raw['_key'].isin(EXCLUDE)].copy()
    if ycol not in raw:
        raise ValueError(f'Expected label {ycol} in {path}; columns={raw.columns.tolist()}')
    raw[ycol] = pd.to_numeric(raw[ycol], errors='raise')
    if not raw[ycol].isin([0,1]).all():
        raise ValueError(f'Invalid binary labels in {path}')
    verify_unique(raw, str(path))
    expected = EXPECTED_COMBINED if kind=='combined' else EXPECTED_SYN
    if len(raw) != expected:
        raise RuntimeError(f'{path}: expected {expected} rows, found {len(raw)}')
    return raw


def load_original_root_targets(summary_path, kind, eligible_keys):
    """Rebuild root targets from unrounded source objectives like the original root runner.

    Saved root_targets CSVs may round floating values; GB regressors can respond
    to tiny changes in near-tied split gains. Preserve original summary row order.
    """
    raw = pd.read_csv(summary_path)
    out = canon_combined(raw) if kind == 'combined' else canon_synthetic(
        raw, 'Synthetic root summary', expected_keys=eligible_keys)
    missing = eligible_keys - set(out['_key'])
    if missing:
        raise RuntimeError(f'{kind} root summary missing {len(missing)} eligible keys')
    out = out.loc[out['_key'].isin(eligible_keys)].copy()
    if len(out) != len(eligible_keys):
        raise RuntimeError('Root summary eligibility/uniqueness mismatch')
    lp = pd.to_numeric(out['lp_root_obj'], errors='raise').to_numpy(float)
    sdp = pd.to_numeric(out['sdp_root_obj'], errors='raise').to_numpy(float)
    if not (np.isfinite(lp).all() and np.isfinite(sdp).all()):
        raise RuntimeError('Non-finite original root objectives')
    out['delta_root'] = (lp - sdp) / (np.abs(lp) + np.abs(sdp) + 1e-3)
    out['y_root'] = (out['delta_root'].to_numpy(float) > -0.01).astype(int)
    return out


def make_model(name):
    if name == 'XGB-C':
        return XGBClassifier(n_estimators=400, learning_rate=0.08, max_depth=6,
            subsample=0.8, colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=1.0,
            random_state=RANDOM_STATE, n_jobs=-1, eval_metric='logloss')
    if name == 'GBC':
        return GradientBoostingClassifier(random_state=RANDOM_STATE)
    if name == 'RFC':
        return RandomForestClassifier(n_estimators=400, max_features='sqrt', class_weight='balanced',
                                      n_jobs=-1, random_state=RANDOM_STATE)
    if name == 'XGB-R':
        return XGBRegressor(n_estimators=400, learning_rate=0.08, max_depth=6,
            subsample=0.8, colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=1.0,
            random_state=RANDOM_STATE, n_jobs=-1)
    if name == 'GBR':
        return GradientBoostingRegressor(random_state=RANDOM_STATE)
    if name == 'RFR':
        return RandomForestRegressor(n_estimators=400, n_jobs=-1, random_state=RANDOM_STATE)
    raise KeyError(name)


def attach(targets, features, columns, label):
    missing = set(targets['_key']) - set(features['_key'])
    if missing: raise RuntimeError(f'{label} missing {len(missing)} feature records, e.g. {sorted(missing)[:8]}')
    out = targets[['_key', 'y' if 'y' in targets.columns else 'y_root', 'delta' if 'delta' in targets.columns else 'delta_root']].merge(
        features[['_key']+columns], on='_key', how='left', validate='one_to_one')
    if len(out)!=len(targets): raise RuntimeError(label+' join changed row count')
    return out


def r_accuracy_tol(y_true_delta, y_pred_delta, eps=0.01):
    # Exact thesis runner definition; keep separate from decision accuracy.
    yt=np.asarray(y_true_delta,dtype=float); yp=np.asarray(y_pred_delta,dtype=float)
    same=np.sign(yt)==np.sign(yp)
    q2=(yt>-eps)&(yt<0)&(yp>0)
    q4=(yt>0)&(yt<eps)&(yp<0)&(yp>-eps)
    return float(np.mean(same|q2|q4))


def metrics(task,y_true,y_pred,delta_true=None,delta_pred=None):
    classification={'accuracy':float(accuracy_score(y_true,y_pred)),
                    'balanced_accuracy':float(balanced_accuracy_score(y_true,y_pred)),
                    'f1_macro':float(f1_score(y_true,y_pred,average='macro',zero_division=0))}
    if task=='classification':return classification
    classification.update({
        'rAccuracy_tol':r_accuracy_tol(delta_true,delta_pred),
        'decision_accuracy':classification['accuracy'],
        'mae':float(mean_absolute_error(delta_true,delta_pred)),
        'rmse':float(np.sqrt(mean_squared_error(delta_true,delta_pred))),
        'r2':float(r2_score(delta_true,delta_pred))})
    return classification


def ref_paths(kind, setting, task, model_name):
    slug='combined_to_combined' if setting=='combined' else 'synthetic_to_combined'
    if kind=='bb':
        rid=(1 if setting=='combined' else 3)+(1 if task=='regression' else 0)
        folder=BB_RUN/f'run_{rid:02d}__{slug}__static__{task}'
        return (folder/f'results.csv',folder/f'predictions_{model_name}.csv',folder/f'model_{model_name}.pkl')
    did=(1 if setting=='combined' else 3)+(1 if task=='regression' else 0)
    folder=ROOT_RUN/f'design_{did:02d}__{slug}__{task}'
    return (folder/'results_all_models.csv',folder/f'preds_{model_name}.csv',folder/f'model_{model_name}.joblib')


def validate_full(kind, setting, task, model, test, X_test, pred, delta_pred, score):
    """Audit against saved results, without disguising a small refit mismatch.

    The ablation control is the freshly refitted Full_64 model; every reduced
    model follows the same training pipeline. Saved original predictions are
    checked independently and any differences are explicitly recorded.
    """
    rp, pp, mp = ref_paths(kind, setting, task, model)
    for path in (rp, pp, mp):
        if not path.exists():
            raise FileNotFoundError(f'Required original output missing: {path}')
    original_results = pd.read_csv(rp)
    original_results = original_results.loc[original_results['model'] == model]
    if len(original_results) != 1:
        raise RuntimeError(f'Expected exactly one original results row for {model}')
    col = 'balanced_accuracy' if task == 'classification' else ('rAccuracy_tol' if kind == 'bb' else 'r_accuracy_tol')
    expected_primary = float(original_results.iloc[0][col])
    actual_primary = score['balanced_accuracy' if task == 'classification' else 'rAccuracy_tol']
    original_csv = pd.read_csv(pp)
    key_col = '_key' if '_key' in original_csv else 'instance_key'
    original_csv = original_csv[[key_col, 'y_pred' if task == 'classification' else 'delta_pred']].rename(
        columns={key_col: '_key', 'y_pred' if task == 'classification' else 'delta_pred': 'original_prediction'})
    new = pd.DataFrame({'_key':test['_key'].to_numpy(), 'refit_prediction':pred if task=='classification' else delta_pred})
    joined = new.merge(original_csv, on='_key', how='outer', validate='one_to_one', indicator=True)
    if len(joined) != len(test) or not joined['_merge'].eq('both').all():
        raise RuntimeError('Saved vs refitted prediction instance keys do not match')
    # Validate saved original model against the saved CSV, by IDENTIFIER, not file row order.
    # The saved CSV is our historical record. A diagnostic mismatch between the old
    # model and CSV is reported prominently, but does not invalidate a new ablation
    # fit or silently change the historical reference predictions.
    old_model = joblib.load(mp)
    old_X = X_test
    old_feature_order = getattr(old_model, 'feature_names_in_', None)
    if old_feature_order is not None:
        old_feature_order = list(old_feature_order)
        if set(old_feature_order) == set(X_test.columns):
            old_X = X_test.loc[:, old_feature_order]
        else:
            raise RuntimeError(f'Saved model feature schema differs from 64 DI features: {mp}')
    original_fitted_pred = old_model.predict(old_X)
    saved_aligned = test[['_key']].merge(original_csv, on='_key', validate='one_to_one')['original_prediction'].to_numpy()
    if task=='classification':
        model_csv_differences = np.abs(original_fitted_pred.astype(float) - saved_aligned.astype(float))
        saved_model_matches_csv = bool(np.array_equal(original_fitted_pred.astype(int), saved_aligned.astype(int)))
        different_instances=int(np.count_nonzero(joined['refit_prediction'].to_numpy(int) != joined['original_prediction'].to_numpy(int)))
        max_abs=float(different_instances>0)
        mean_abs=float(different_instances / len(test))
        decision_disagreements=different_instances
        reproduce=(different_instances == 0)
    else:
        model_csv_differences = np.abs(original_fitted_pred.astype(float) - saved_aligned.astype(float))
        saved_model_matches_csv = bool(np.allclose(original_fitted_pred.astype(float), saved_aligned.astype(float), rtol=1e-8, atol=1e-10))
        differences=np.abs(joined['refit_prediction'].to_numpy(float)-joined['original_prediction'].to_numpy(float))
        max_abs=float(differences.max())
        mean_abs=float(differences.mean())
        decision_disagreements=int(np.count_nonzero(
            (joined['refit_prediction'].to_numpy(float)>-0.01) != (joined['original_prediction'].to_numpy(float)>-0.01)))
        different_instances=int(np.count_nonzero(differences>1e-8))
        reproduce=bool(np.allclose(joined['refit_prediction'].to_numpy(float), joined['original_prediction'].to_numpy(float), rtol=1e-6, atol=1e-8))
    if not saved_model_matches_csv:
        print(f'  AUDIT WARNING: saved model predictions differ from saved CSV for {kind}/{setting}/{task}/{model}; max abs={model_csv_differences.max():.6g}. Check baseline audit CSV. Historical CSV kept unchanged.', flush=True)
    record={'kind':kind,'setting':setting,'task':task,'model':model,
        'matches_saved_original':reproduce, 'saved_model_matches_saved_csv':saved_model_matches_csv,
        'saved_model_vs_csv_max_abs_diff':float(model_csv_differences.max()),
        'saved_model_vs_csv_n_different':int(np.count_nonzero(model_csv_differences > 1e-8)),
        'saved_primary_score':expected_primary,
        'refit_primary_score':float(actual_primary), 'primary_metric_abs_diff':abs(expected_primary-actual_primary),
        'max_abs_prediction_diff':max_abs,'mean_abs_prediction_diff':mean_abs,
        'n_different_prediction_values':different_instances,'n_different_decisions':decision_disagreements}
    if abs(expected_primary-actual_primary)>0.015:
        raise RuntimeError(f'Full-feature refit metric differs materially from original: {record}')
    if not reproduce:
        print('  BASELINE AUDIT WARNING: refit not identical to original; '
              f'primary gap={record["primary_metric_abs_diff"]:.6g}, '
              f'max delta gap={max_abs:.6g}, decision mismatches={decision_disagreements}', flush=True)
    else:
        print('  ORIGINAL BASELINE PREDICTIONS MATCH', flush=True)
    return record


def main():
    required=[COMBINED_STATIC,SYN_STATIC,COMBINED_BB_TARGET,SYN_BB_TARGET,BB_SPLIT,
              ROOT_COMBINED_TARGET,ROOT_SYN_TARGET,ROOT_COMBINED_SUMMARY,ROOT_SYN_SUMMARY]
    missing=[str(p) for p in required if not p.exists()]
    if missing:raise FileNotFoundError('Missing inputs:\n'+'\n'.join(missing))
    bb_cols_path=BB_RUN/'run_01__combined_to_combined__static__classification'/'feature_columns.csv'
    root_cols_path=ROOT_RUN/'design_01__combined_to_combined__classification'/'feature_columns.csv'
    cols=pd.read_csv(bb_cols_path)['feature'].astype(str).tolist()
    root_cols=pd.read_csv(root_cols_path)['feature'].astype(str).tolist()
    if len(cols)!=64 or cols!=root_cols or len(set(cols))!=64:raise RuntimeError('64-feature schema mismatch')
    groups={'objective_A0':OBJECTIVE_SPECTRAL,'constraint_h':CONSTRAINT_FEATURES,
            'structure_g_comm':STRUCTURAL_FEATURES,'bounds':['BoundsExist']}
    partition=[x for vals in groups.values() for x in vals]
    if len(partition)!=64 or set(partition)!=set(cols) or len(set(partition))!=64:
        raise RuntimeError('Feature group partition mismatch')
    for name,removed in ABLATIONS.items():
        if len(removed)!=len(set(removed)) or set(removed)-set(cols) or len(cols)==len(removed):
            raise RuntimeError(f'Invalid ablation {name}')
    targets={}
    for kind in ['bb','root']:
        ycol='y' if kind=='bb' else 'y_root'
        ct=load_targets(COMBINED_BB_TARGET if kind=='bb' else ROOT_COMBINED_TARGET,'combined',ycol)
        st=load_targets(SYN_BB_TARGET if kind=='bb' else ROOT_SYN_TARGET,'synthetic',ycol)
        targets[kind]=(ct,st,ycol)
    if set(targets['bb'][0]['_key'])!=set(targets['root'][0]['_key']):raise RuntimeError('Combined root/B&B sets differ')
    if set(targets['bb'][1]['_key'])!=set(targets['root'][1]['_key']):raise RuntimeError('Synthetic root/B&B sets differ')
    # Root original runner computes delta_root from raw summary objectives before saving CSV.
    # Recompute the target to prevent tiny CSV roundtrips from changing fitted GBR splits.
    targets['root']=(load_original_root_targets(ROOT_COMBINED_SUMMARY,'combined',set(targets['bb'][0]['_key'])),
                     load_original_root_targets(ROOT_SYN_SUMMARY,'synthetic',set(targets['bb'][1]['_key'])), 'y_root')
    cf=load_features(COMBINED_STATIC,'combined')
    sf=load_features(SYN_STATIC,'synthetic',set(targets['bb'][1]['_key']))
    for label,frame in [('Combined',cf),('Synthetic',sf)]:
        if set(cols)-set(frame.columns):raise RuntimeError(label+' feature columns missing')
    data={}
    for kind in ['bb','root']:
        ct,st,ycol=targets[kind]
        data[kind]=(attach(ct,cf,cols,kind+' combined'),attach(st,sf,cols,kind+' synthetic'),ycol)
    split=pd.read_csv(BB_SPLIT)
    if not {'_key','split'}.issubset(split):raise RuntimeError('Invalid split manifest')
    split['_key']=split['_key'].astype(str).str.strip()
    if len(split)!=280 or split['_key'].duplicated().any() or set(split['_key'])!=set(data['bb'][0]['_key']):
        raise RuntimeError('Invalid Combined split instance set')
    train_keys=set(split.loc[split['split']=='train','_key']); test_keys=set(split.loc[split['split']=='test','_key'])
    if len(train_keys)!=177 or len(test_keys)!=103 or train_keys&test_keys:raise RuntimeError('Invalid train/test split')
    OUT.mkdir(parents=True,exist_ok=False)
    pd.DataFrame([{'feature':c,'family':next(g for g,fs in groups.items() if c in fs)} for c in cols]).to_csv(OUT/'feature_groups_64.csv',index=False)
    pd.DataFrame([{'variant':n,'n_removed':len(fs),'removed_features':';'.join(fs)} for n,fs in ABLATIONS.items()]).to_csv(OUT/'ablation_definitions.csv',index=False)
    all_rows=[]
    for exp,kind,setting,cls_model,reg_model,tier in PLAN:
        cd,sd,ycol=data[kind]
        train=cd.loc[cd['_key'].isin(train_keys)].copy() if setting=='combined' else sd.copy()
        test=cd.loc[cd['_key'].isin(test_keys)].copy() if setting=='combined' else cd.copy()
        ytr=train[ycol].to_numpy(int); yte=test[ycol].to_numpy(int)
        dcol='delta' if kind=='bb' else 'delta_root'
        dtr=train[dcol].to_numpy(float); dte=test[dcol].to_numpy(float)
        for task,model_name in [('classification',cls_model),('regression',reg_model)]:
            name=exp+'_'+task.title()
            print(f'\n{name} | {model_name} | train={len(train)} test={len(test)}',flush=True)
            run_rows=[]
            for variant,removed in ABLATIONS.items():
                kept=[c for c in cols if c not in removed]
                Xt=train[kept].apply(pd.to_numeric,errors='coerce').replace([np.inf,-np.inf],np.nan)
                Xv=test[kept].apply(pd.to_numeric,errors='coerce').replace([np.inf,-np.inf],np.nan)
                pipe=Pipeline([('imputer',SimpleImputer(strategy='median',keep_empty_features=True)),('model',make_model(model_name))])
                pipe.fit(Xt,ytr if task=='classification' else dtr)
                raw_pred=pipe.predict(Xv)
                delta_pred=raw_pred.astype(float) if task=='regression' else None
                pred=(delta_pred>-.01).astype(int) if task=='regression' else raw_pred.astype(int)
                score=metrics(task,yte,pred,dte,delta_pred)
                row={'experiment':name,'method':kind,'training_setting':setting,'task':task,'tier':tier,
                     'model':model_name,'variant':variant,'n_train':len(train),'n_test':len(test),
                     'n_features':len(kept),'n_removed':len(removed),**score}
                if variant=='Full_64':
                    audit=validate_full(kind,setting,task,model_name,test,Xv,pred,delta_pred,score)
                    row.update(audit)
                    pd.DataFrame([audit]).to_csv(OUT/f'{name}__baseline_audit.csv',index=False)
                    marker=' [BASELINE CHECKED]' 
                else:marker=''
                run_rows.append(row);all_rows.append(row)
                pred_df=pd.DataFrame({'_key':test['_key'].values,'y_true':yte,'y_pred':pred,'delta_true':dte})
                if task=='regression':pred_df['delta_pred']=delta_pred
                else:pred_df['prob_LP']=pipe.predict_proba(Xv)[:,1]
                pred_df.to_csv(OUT/f'{name}__{variant}__predictions.csv',index=False)
                main_metric='balanced_accuracy' if task=='classification' else 'rAccuracy_tol'
                print(f'  {variant:29s} {main_metric}={score[main_metric]:.4f}{marker}',flush=True)
            df=pd.DataFrame(run_rows)
            base=df.loc[df.variant=='Full_64'].iloc[0]
            metrics_to_compare=['accuracy','balanced_accuracy','f1_macro'] if task=='classification' else ['rAccuracy_tol','decision_accuracy','mae','rmse','r2']
            for row in run_rows:
                for metric in metrics_to_compare:
                    row['drop_'+metric]=float(base[metric]-row[metric])
                row['primary_metric']='balanced_accuracy' if task=='classification' else 'rAccuracy_tol'
                row['primary_score']=row[row['primary_metric']]
                row['drop_primary']=row['drop_'+row['primary_metric']]
                row['primary_score_percentage_points']=100*row['primary_score']
                row['drop_primary_percentage_points']=100*row['drop_primary']
            pd.DataFrame(run_rows).to_csv(OUT/f'{name}.csv',index=False)
            pd.DataFrame(all_rows).to_csv(OUT/'all_ablation_results.csv',index=False)
    results=pd.DataFrame(all_rows)
    for task,metric in [('classification','balanced_accuracy'),('regression','rAccuracy_tol')]:
        sub=results.loc[results.task==task]
        sub.pivot(index=['experiment','model'],columns='variant',values='drop_'+metric).to_csv(OUT/f'{task}_primary_drops_wide.csv')
        for tier in ['primary','robustness']:
            rows=sub.loc[sub.tier==tier]
            pv=rows.pivot(index='experiment',columns='variant',values='drop_'+metric)
            pv=pv[[k for k in ABLATIONS if k!='Full_64']]
            ax=pv.plot(kind='bar',figsize=(max(12,len(pv.columns)*1.2),6),width=0.85)
            ax.axhline(0,color='black',linewidth=.7)
            ax.set_ylabel('Drop in balanced accuracy' if task=='classification' else 'Drop in rAccuracy_tol')
            ax.set_xlabel('');ax.tick_params(axis='x',rotation=20)
            ax.set_title(f'{task.title()} | {tier} | held-out group ablations')
            ax.figure.tight_layout();ax.figure.savefig(OUT/f'{task}_{tier}_ablation.png',dpi=180);plt.close(ax.figure)
    summary=results[['experiment','method','training_setting','task','model','tier','variant','n_removed',
                     'n_train','n_test','primary_metric','primary_score','drop_primary',
                     'primary_score_percentage_points','drop_primary_percentage_points']]
    summary.to_csv(OUT/'paper_primary_metric_comparison.csv',index=False)
    print('\nCOMPLETE:',OUT)
    print(summary[['experiment','variant','primary_metric','primary_score','drop_primary']].to_string(index=False))

if __name__=='__main__':
    main()


Synthetic targets: matched via instance_key, overlap=6069
Synthetic targets: matched via _key, overlap=6069
Synthetic root summary: matched via instance_key, overlap=6069
Synthetic static: matched via id, overlap=6069

BB_Combined_Classification | GBC | train=177 test=103
  ORIGINAL BASELINE PREDICTIONS MATCH
  Full_64                       balanced_accuracy=0.8998 [BASELINE CHECKED]
  No_objective_A0               balanced_accuracy=0.8915
  No_constraint_h               balanced_accuracy=0.8590
  No_structure_g_comm           balanced_accuracy=0.8998
  No_BoundsExist                balanced_accuracy=0.8998
  No_objective_extremes         balanced_accuracy=0.8721
  No_objective_shape            balanced_accuracy=0.8581
  No_h_central                  balanced_accuracy=0.8590
  No_h_dispersion               balanced_accuracy=0.8896
  No_objective_or_h             balanced_accuracy=0.8915
  Only_objective_and_bounds     balanced_accuracy=0.8683
  Only_constraint_and_bounds    balanced_ac